In [1]:
import os
import sys
import re
import glob
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb
import faiss
import pypdf

/Users/user/Downloads/Clouddesk Amdari/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATASET_DIR = "clouddesk_dataset"

source_summary = []
for folder in os.listdir(DATASET_DIR):
    folder_path = os.path.join(DATASET_DIR, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        source_summary.append({
            'Folder': folder,
            'File Count': len(files),
            'Sample Files': ', '.join(files[:2])
        })

df_audit = pd.DataFrame(source_summary)
try: display(df_audit)
except NameError: print(df_audit)

,Folder,File Count,Sample Files
0,engineering_runbooks,3,"runbook-webhook-delivery-failures.md, runbook-..."
1,release_notes,3,"RN-3.5.0_release-notes.pdf, RN-3.4.0_release-n..."
2,api_documentation,4,"authentication.md, webhooks-reference.md"
3,support_tickets,1,previous_support_tickets.csv
4,help_center_articles,5,"HC-101_integrating-clouddesk-with-slack.pdf, H..."


In [5]:
class DocumentChunk:
    def __init__(self, text: str, metadata: dict):
        self.page_content = text.strip()
        self.metadata = metadata

    def __repr__(self):
        return f"<Chunk id={self.metadata.get('chunk_id')} type={self.metadata.get('source_type')} title='{self.metadata.get('title')}'>"
    

In [6]:
def parse_markdown(filepath: str, source_type: str) -> tuple:
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    meta = {
        'source_type': source_type,
        'title': os.path.splitext(os.path.basename(filepath))[0].replace('-', ' ').title(),
        'last_updated': '2026-09-15',
        'version': 'v3.6.0',
        'source_file': os.path.basename(filepath)
    }
    body = content
    if content.startswith('---'):
        parts = content.split('---', 2)
        if len(parts) >= 3:
            body = parts[2]
            for line in parts[1].strip().split('\n'):
                if ':' in line:
                    k, v = line.split(':', 1)
                    meta[k.strip().lower()] = v.strip().strip('"\'')
    for line in body.split('\n'):
        if line.startswith('# '):
            meta['title'] = line.replace('#', ' ').strip()
            break
    return meta, body

In [7]:
def parse_pdf(filepath: str, source_type: str) -> tuple:
    reader = pypdf.PdfReader(filepath)
    text = '\n'.join((p.extract_text() or '') for p in reader.pages)
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    header = ' '.join(lines[:4])

    meta = {
        'source_type': source_type,
        'title': lines[0] if lines else os.path.splitext(os.path.basename(filepath))[0],
        'last_updated': 'unknown',
        'version': 'unknown',
        'source_file': os.path.basename(filepath)
    }
    if (m := re.search(r'Doc ID:\s*([\w\.\-]+)', header)): meta['doc_id'] = m.group(1)
    if (m := re.search(r'Version:\s*(v?[\d\.]+)', header)): meta['version'] = m.group(1)
    if (m := re.search(r'(?:Last updated|Release date):\s*(\d{4}-\d{2}-\d{2})', header, re.I)): meta['last_updated'] = m.group(1)
    return meta, text

In [8]:
def find_dir(*names):
    for n in names:
        p = os.path.join(DATASET_DIR, n)
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f"None of {names} found under {DATASET_DIR}")

def split_text(text: str, max_chars: int = 1000, overlap: int = 150) -> list:
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    out, cur = [], ''
    for line in lines:
        if cur and len(cur) + len(line) + 1 > max_chars:
            out.append(cur)
            tail = cur[-overlap:]
            tail = tail[tail.find(' ') + 1:] if ' ' in tail else tail
            cur = tail + '\n' + line
        else:
            cur = (cur + '\n' + line) if cur else line
    if cur:
        out.append(cur)
    return out

chunks, counter = [], 0

def add_chunks(text: str, meta: dict, prefix: str):
    global counter
    for part in split_text(text):
        if len(part.strip()) < 40:
            continue
        counter += 1
        m = meta.copy()
        m['chunk_id'] = f'{prefix}_{counter:04d}'
        chunks.append(DocumentChunk(f"{meta['title']}\n\n{part}", m))   # title prefix helps retrieval

for fp in glob.glob(os.path.join(find_dir('api_documentation'), '*.md')):
    meta, body = parse_markdown(fp, 'api_documentation')
    for sec in body.split('\n## '):
        add_chunks(sec, meta, 'api_doc')

for fp in glob.glob(os.path.join(find_dir('engineering_runbooks', 'engineering_runbook'), '*.md')):
    meta, body = parse_markdown(fp, 'engineering_runbook')
    for sec in body.split('\n## '):
        add_chunks(sec, meta, 'runbook')

for fp in glob.glob(os.path.join(find_dir('help_center_articles'), '*.pdf')):
    meta, text = parse_pdf(fp, 'help_center_article')
    add_chunks(text, meta, 'hc_art')

for fp in glob.glob(os.path.join(find_dir('release_notes'), '*.pdf')):
    meta, text = parse_pdf(fp, 'release_notes')
    add_chunks(text, meta, 'rel_note')

csv_p = os.path.join(DATASET_DIR, 'support_tickets', 'previous_support_tickets.csv')
df_t = pd.read_csv(csv_p)
for _, r in df_t.iterrows():
    tid = str(r['ticket_id'])
    content = f"Ticket [{tid}] Subject: {r['subject']}\nQuestion: {r['customer_question']}\nResolution: {r['agent_answer']}"
    m = {
        'source_type': 'support_ticket',
        'title': f"Ticket {tid}: {r['subject']}",
        'chunk_id': f'ticket_{tid}',
        'last_updated': str(r['created_at']),
        'version': 'n/a',
        'source_file': 'previous_support_tickets.csv',
        'source_type_reference': str(r.get('source_type_reference', '')),
        'source_title_reference': str(r.get('source_title_reference', ''))
    }
    chunks.append(DocumentChunk(content, m))
    counter += 1

In [9]:
len(chunks)

111

In [10]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c.page_content for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

print('Embedding matrix shape', embeddings.shape)
print(f'vector dimension: {embeddings.shape[1]}')

Batches: 100%|██████████| 4/4 [00:01<00:00,  2.16it/s]

Embedding matrix shape (111, 384)
vector dimension: 384


In [11]:
VECTOR_DB_DIR = 'vector_db'
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=os.path.join(VECTOR_DB_DIR, 'chroma'))
try:
    chroma_client.delete_collection('clouddesk_support_kb')   # rebuild from scratch (old chunks were wrong)
except Exception:
    pass
collection = chroma_client.create_collection(
    name='clouddesk_support_kb',
    metadata={'hnsw:space': 'cosine'}
)

collection.add(
    ids=[c.metadata['chunk_id'] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=[c.page_content for c in chunks],
    metadatas=[c.metadata for c in chunks]
)
print(f'ChromaDB Indexed: {collection.count()} vectors')

ChromaDB Indexed: 111 vectors


In [ ]:
# FAISS

In [12]:
%pip install pinecone


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
# PINECONE  (index names may only contain lowercase letters, numbers and hyphens)
pinecone_key = os.environ.get('PINECONE_API_KEY')
if pinecone_key:
    try:
        from pinecone import Pinecone, ServerlessSpec

        pc = Pinecone(api_key=pinecone_key)
        index_name = os.environ.get('PINECONE_INDEX_NAME', 'clouddesk-support-rag')
        existing = [idx.name for idx in pc.list_indexes()]

        if index_name not in existing:
            pc.create_index(
                name=index_name,
                dimension=384,
                metric='cosine',
                spec=ServerlessSpec(cloud='aws', region='us-east-1')
            )

        pc_index = pc.Index(index_name)
        try:
            pc_index.delete(delete_all=True)     # drop stale vectors from earlier faulty runs
        except Exception:
            pass

        vectors_to_upsert = []
        for c, emb in zip(chunks, embeddings):
            meta = {k: str(v) for k, v in c.metadata.items()}
            meta['text'] = c.page_content[:8000]   # full chunk text; the retrieval script reads 'text'
            vectors_to_upsert.append((c.metadata['chunk_id'], emb.tolist(), meta))

        for i in range(0, len(vectors_to_upsert), 50):
            pc_index.upsert(vectors=vectors_to_upsert[i:i+50])

        print(f'Successfully indexed {len(vectors_to_upsert)} chunks into pinecone')
    except Exception as e:
        print(f'Pinecone indexing notice: {e}')
else:
    print('PINECONE_API_KEY not found in environment')

Successfully indexed 111 chunks into pinecone


In [14]:
with open(os.path.join(VECTOR_DB_DIR, 'chunks_store.json'), 'w', encoding='utf-8') as f:
    json.dump([{'page_content': c.page_content, 'metadata': c.metadata} for c in chunks], f, indent=2)

In [15]:
# Sanity check: every source type should appear, including help_center_article and release_notes
print(pd.Series([c.metadata['source_type'] for c in chunks]).value_counts())

support_ticket         43
api_documentation      29
engineering_runbook    22
help_center_article    13
release_notes           4
Name: count, dtype: int64


In [16]:
[c.metadata['title'] for c in chunks if c.metadata['source_type'] == 'help_center_article'][:3]

['Integrating CloudDesk with Slack',
 'Integrating CloudDesk with Slack',
 'Integrating CloudDesk with Slack']

In [17]:
print(pd.Series([c.metadata['source_type'] for c in chunks]).value_counts())

support_ticket         43
api_documentation      29
engineering_runbook    22
help_center_article    13
release_notes           4
Name: count, dtype: int64
